# Customer Churn Prediction Using PySpark

## Overview

This project applies Apache Spark's machine learning API to customer churn
prediction using two customer datasets.

The analysis includes:

- Data loading and exploratory inspection
- Missing-value handling
- Numerical and categorical feature preprocessing
- Train/test splitting
- Logistic Regression
- Random Forest
- Gradient-Boosted Trees
- Model evaluation using Accuracy, Precision, Recall, F1, and ROC-AUC
- Confusion-matrix analysis
- Logistic Regression coefficient analysis
- Final model comparison

The objective is to identify which classification approach provides the
strongest predictive performance for customer churn.

In [3]:
%pip install pyspark

     ---------------------------------------- 0.0/450.1 MB ? eta -:--:--
     --------------------------------------- 2.9/450.1 MB 16.0 MB/s eta 0:00:28
     --------------------------------------- 3.1/450.1 MB 16.5 MB/s eta 0:00:28
      -------------------------------------- 8.4/450.1 MB 14.2 MB/s eta 0:00:32
     - ------------------------------------ 12.6/450.1 MB 16.6 MB/s eta 0:00:27
     - ------------------------------------ 12.6/450.1 MB 16.6 MB/s eta 0:00:27
     - ------------------------------------ 18.6/450.1 MB 15.1 MB/s eta 0:00:29
     -- ----------------------------------- 24.4/450.1 MB 17.0 MB/s eta 0:00:25
     -- ----------------------------------- 29.9/450.1 MB 18.2 MB/s eta 0:00:24
     -- ----------------------------------- 29.9/450.1 MB 18.2 MB/s eta 0:00:24
     -- ----------------------------------- 34.1/450.1 MB 16.5 MB/s eta 0:00:26
     --- ---------------------------------- 38.8/450.1 MB 17.1 MB/s eta 0:00:25
     --- ---------------------------------- 45.

In [8]:
# HW3 - Customer Churn Prediction using Spark API

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

# Create Spark session
spark = (
    SparkSession.builder
    .appName("Customer Churn Prediction - HW3")
    .getOrCreate()
)

print("Spark version:", spark.version)

Spark version: 4.2.0


In [9]:
# Load the datasets

dataset1_path = "dataset1_HW1.csv"
dataset2_path = "dataset2_HW1.csv"

df1 = spark.read.csv(dataset1_path, header=True, inferSchema=True)
df2 = spark.read.csv(dataset2_path, header=True, inferSchema=True)

print("Dataset 1:")
print("Rows:", df1.count())
print("Columns:", len(df1.columns))

print("\nDataset 2:")
print("Rows:", df2.count())
print("Columns:", len(df2.columns))

Dataset 1:
Rows: 40000
Columns: 14

Dataset 2:
Rows: 50000
Columns: 14


In [10]:
# Inspect the datasets

print("===== DATASET 1 =====")
df1.printSchema()

print("\nFirst 5 rows:")
df1.show(5, truncate=False)

print("\n===== DATASET 2 =====")
df2.printSchema()

print("\nFirst 5 rows:")
df2.show(5, truncate=False)

===== DATASET 1 =====
root
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- region: string (nullable = true)
 |-- income: double (nullable = true)
 |-- tenure_months: integer (nullable = true)
 |-- monthly_spend: double (nullable = true)
 |-- website_visits: integer (nullable = true)
 |-- support_tickets: integer (nullable = true)
 |-- email_click_rate: double (nullable = true)
 |-- loyalty_score: double (nullable = true)
 |-- campaign_type: string (nullable = true)
 |-- discount_used: integer (nullable = true)
 |-- last_campaign_days: integer (nullable = true)
 |-- churn: integer (nullable = true)


First 5 rows:
+---+------+------+--------+-------------+-------------+--------------+---------------+----------------+-------------+-------------+-------------+------------------+-----+
|age|gender|region|income  |tenure_months|monthly_spend|website_visits|support_tickets|email_click_rate|loyalty_score|campaign_type|discount_used|last_campaign_days|churn|
+---

In [12]:
# Separate features and target

target = "churn"

feature_cols = [c for c in df1.columns if c != target]

print("Target:", target)
print("Features:", feature_cols)

Target: churn
Features: ['age', 'gender', 'region', 'income', 'tenure_months', 'monthly_spend', 'website_visits', 'support_tickets', 'email_click_rate', 'loyalty_score', 'campaign_type', 'discount_used', 'last_campaign_days']


In [13]:
# Identify numeric and categorical columns

numeric_cols = [
    "age", "income", "tenure_months", "monthly_spend",
    "website_visits", "support_tickets", "email_click_rate",
    "loyalty_score", "discount_used", "last_campaign_days"
]

categorical_cols = [
    "gender", "region", "campaign_type"
]

print("Numeric columns:", numeric_cols)
print("Categorical columns:", categorical_cols)

Numeric columns: ['age', 'income', 'tenure_months', 'monthly_spend', 'website_visits', 'support_tickets', 'email_click_rate', 'loyalty_score', 'discount_used', 'last_campaign_days']
Categorical columns: ['gender', 'region', 'campaign_type']


In [15]:
# Check missing values

print("Missing values - Dataset 1")
df1.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df1.columns
]).show()

print("Missing values - Dataset 2")
df2.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df2.columns
]).show()

Missing values - Dataset 1
+---+------+------+------+-------------+-------------+--------------+---------------+----------------+-------------+-------------+-------------+------------------+-----+
|age|gender|region|income|tenure_months|monthly_spend|website_visits|support_tickets|email_click_rate|loyalty_score|campaign_type|discount_used|last_campaign_days|churn|
+---+------+------+------+-------------+-------------+--------------+---------------+----------------+-------------+-------------+-------------+------------------+-----+
|  0|     0|     0|     0|            0|            0|             0|              0|               0|            0|            0|            0|                 0|    0|
+---+------+------+------+-------------+-------------+--------------+---------------+----------------+-------------+-------------+-------------+------------------+-----+

Missing values - Dataset 2
+---+------+------+------+-------------+-------------+--------------+---------------+----------

In [16]:
#  Train/test split

train1, test1 = df1.randomSplit([0.8, 0.2], seed=42)
train2, test2 = df2.randomSplit([0.8, 0.2], seed=42)

print("Dataset 1 - Train:", train1.count(), "Test:", test1.count())
print("Dataset 2 - Train:", train2.count(), "Test:", test2.count())

Dataset 1 - Train: 32011 Test: 7989
Dataset 2 - Train: 39948 Test: 10052


In [17]:
# Spark preprocessing pipeline

from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    StringIndexer,
    OneHotEncoder,
    VectorAssembler,
    StandardScaler,
    Imputer
)

# Handle missing numeric values
imputer = Imputer(
    inputCols=numeric_cols,
    outputCols=[c + "_imputed" for c in numeric_cols]
)

imputed_numeric_cols = [c + "_imputed" for c in numeric_cols]

# Convert categorical columns to indexes
indexers = [
    StringIndexer(
        inputCol=c,
        outputCol=c + "_index",
        handleInvalid="keep"
    )
    for c in categorical_cols
]

indexed_categorical_cols = [c + "_index" for c in categorical_cols]

# One-hot encode categorical variables
encoder = OneHotEncoder(
    inputCols=indexed_categorical_cols,
    outputCols=[c + "_encoded" for c in categorical_cols]
)

encoded_categorical_cols = [c + "_encoded" for c in categorical_cols]

# Combine all features
assembler = VectorAssembler(
    inputCols=imputed_numeric_cols + encoded_categorical_cols,
    outputCol="features_raw"
)

# Scale features
scaler = StandardScaler(
    inputCol="features_raw",
    outputCol="features",
    withMean=False,
    withStd=True
)

preprocessing_pipeline = Pipeline(stages=[
    imputer,
    *indexers,
    encoder,
    assembler,
    scaler
])

print("Preprocessing pipeline created successfully.")

Preprocessing pipeline created successfully.


In [18]:
#  Define 3 machine learning models

from pyspark.ml.classification import (
    LogisticRegression,
    RandomForestClassifier,
    GBTClassifier
)

lr = LogisticRegression(
    featuresCol="features",
    labelCol="churn",
    maxIter=100
)

rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="churn",
    numTrees=100,
    seed=42
)

gbt = GBTClassifier(
    featuresCol="features",
    labelCol="churn",
    maxIter=100,
    seed=42
)

print("Models created:")
print("1. Logistic Regression")
print("2. Random Forest")
print("3. Gradient-Boosted Trees")

Models created:
1. Logistic Regression
2. Random Forest
3. Gradient-Boosted Trees


In [19]:
#  Train the three models on both datasets

models = {
    "Logistic Regression": lr,
    "Random Forest": rf,
    "Gradient-Boosted Trees": gbt
}

results = {}

for dataset_name, train_df, test_df in [
    ("Dataset 1", train1, test1),
    ("Dataset 2", train2, test2)
]:
    
    print(f"\n===== {dataset_name} =====")
    
    # Fit preprocessing on training data only
    preprocessing_model = preprocessing_pipeline.fit(train_df)
    
    # Transform train and test data
    train_prepared = preprocessing_model.transform(train_df)
    test_prepared = preprocessing_model.transform(test_df)
    
    dataset_results = {}
    
    for model_name, model in models.items():
        print(f"\nTraining {model_name}...")
        
        fitted_model = model.fit(train_prepared)
        
        predictions = fitted_model.transform(test_prepared)
        
        dataset_results[model_name] = {
            "preprocessing_model": preprocessing_model,
            "model": fitted_model,
            "predictions": predictions
        }
        
        print(f"{model_name} completed.")
    
    results[dataset_name] = dataset_results

print("\nAll models trained successfully.")


===== Dataset 1 =====

Training Logistic Regression...
Logistic Regression completed.

Training Random Forest...
Random Forest completed.

Training Gradient-Boosted Trees...
Gradient-Boosted Trees completed.

===== Dataset 2 =====

Training Logistic Regression...
Logistic Regression completed.

Training Random Forest...
Random Forest completed.

Training Gradient-Boosted Trees...
Gradient-Boosted Trees completed.

All models trained successfully.


In [21]:
#  Evaluate model performance

from pyspark.ml.evaluation import (
    MulticlassClassificationEvaluator,
    BinaryClassificationEvaluator
)

evaluation_results = []

for dataset_name, dataset_results in results.items():
    
    for model_name, model_info in dataset_results.items():
        
        predictions = model_info["predictions"]
        
        accuracy = MulticlassClassificationEvaluator(
            labelCol="churn",
            predictionCol="prediction",
            metricName="accuracy"
        ).evaluate(predictions)
        
        precision = MulticlassClassificationEvaluator(
            labelCol="churn",
            predictionCol="prediction",
            metricName="weightedPrecision"
        ).evaluate(predictions)
        
        recall = MulticlassClassificationEvaluator(
            labelCol="churn",
            predictionCol="prediction",
            metricName="weightedRecall"
        ).evaluate(predictions)
        
        f1 = MulticlassClassificationEvaluator(
            labelCol="churn",
            predictionCol="prediction",
            metricName="f1"
        ).evaluate(predictions)
        
        roc_auc = BinaryClassificationEvaluator(
            labelCol="churn",
            rawPredictionCol="rawPrediction",
            metricName="areaUnderROC"
        ).evaluate(predictions)
        
        evaluation_results.append({
            "Dataset": dataset_name,
            "Model": model_name,
            "Accuracy": accuracy,
            "Precision": precision,
            "Recall": recall,
            "F1": f1,
            "ROC_AUC": roc_auc
        })

print("Evaluation completed.")

Evaluation completed.


In [22]:
#  Compare model performance

import pandas as pd

results_df = pd.DataFrame(evaluation_results)

display(
    results_df.sort_values(
        ["Dataset", "ROC_AUC"],
        ascending=[True, False]
    )
)

,Dataset,Model,Accuracy,Precision,Recall,F1,ROC_AUC
0,Dataset 1,Logistic Regression,0.810740,0.810689,0.810740,0.810687,0.896021
2,Dataset 1,Gradient-Boosted Trees,0.803355,0.803305,0.803355,0.803314,0.888460
1,Dataset 1,Random Forest,0.794467,0.795334,0.794467,0.793902,0.875829
3,Dataset 2,Logistic Regression,0.939912,0.937465,0.939912,0.938208,0.968957
5,Dataset 2,Gradient-Boosted Trees,0.936132,0.933847,0.936132,0.934685,0.965763
4,Dataset 2,Random Forest,0.924990,0.923488,0.924990,0.914168,0.948949


In [23]:
#  Select the best model for each dataset

best_models = {}

for dataset_name in results_df["Dataset"].unique():
    
    dataset_results = results_df[
        results_df["Dataset"] == dataset_name
    ]
    
    best_row = dataset_results.loc[
        dataset_results["ROC_AUC"].idxmax()
    ]
    
    best_models[dataset_name] = best_row["Model"]
    
    print(
        f"{dataset_name}: "
        f"{best_row['Model']} "
        f"(ROC-AUC = {best_row['ROC_AUC']:.4f})"
    )

Dataset 1: Logistic Regression (ROC-AUC = 0.8960)
Dataset 2: Logistic Regression (ROC-AUC = 0.9690)


In [24]:
#  Confusion matrices for the best models

for dataset_name, best_model_name in best_models.items():
    
    predictions = results[dataset_name][best_model_name]["predictions"]
    
    print(f"\n===== {dataset_name} - {best_model_name} =====")
    
    confusion_matrix = (
        predictions
        .groupBy("churn", "prediction")
        .count()
        .orderBy("churn", "prediction")
    )
    
    confusion_matrix.show()


===== Dataset 1 - Logistic Regression =====
+-----+----------+-----+
|churn|prediction|count|
+-----+----------+-----+
|    0|       0.0| 3047|
|    0|       1.0|  779|
|    1|       0.0|  733|
|    1|       1.0| 3430|
+-----+----------+-----+


===== Dataset 2 - Logistic Regression =====
+-----+----------+-----+
|churn|prediction|count|
+-----+----------+-----+
|    0|       0.0|  901|
|    0|       1.0|  381|
|    1|       0.0|  223|
|    1|       1.0| 8547|
+-----+----------+-----+



In [25]:
#  Feature importance for the best models

for dataset_name, best_model_name in best_models.items():

    fitted_model = results[dataset_name][best_model_name]["model"]

    print(f"\n===== {dataset_name} =====")
    print(f"Model: {best_model_name}")

    coefficients = fitted_model.coefficients

    print("Number of coefficients:", len(coefficients))
    print("Coefficients:", coefficients)


===== Dataset 1 =====
Model: Logistic Regression
Number of coefficients: 21
Coefficients: [-0.012747763040582314,0.009195200154569115,-0.7102062863414049,1.73964522849017,-0.2402444385658958,0.035632952789726224,-0.4174954743986128,-0.3742750564970675,0.05301994916620876,1.5573062207478865,0.004129155766652062,-0.001823710238364828,-0.005761235290099591,0.0011403373870179834,-0.015372143312699168,-0.0005549138163807693,0.01488807798801023,0.011734994634163902,0.00014903015941723134,0.005164081361479345,-0.017162000243547408]

===== Dataset 2 =====
Model: Logistic Regression
Number of coefficients: 21
Coefficients: [0.03709595543095339,-0.036172406209102385,-0.6429935420245841,4.250145578731877,-0.29588722367258585,0.040451787855855,-0.403692174049599,-0.3078167817241413,0.11524419940836143,1.600814320798299,-0.004225543835162164,0.00455277259645549,-0.0008162923195920037,-0.02295794222010553,0.018844185797247816,0.01523462372069275,-0.010945713258606186,-0.03303859957512859,0.01833108

In [31]:
#  Extract final feature names from the preprocessing pipeline

stages = preprocessing_pipeline.getStages()

# Get the VectorAssembler
assembler = stages[5]

# Original inputs to the assembler
assembler_inputs = assembler.getInputCols()

print("VectorAssembler inputs:")
for i, col in enumerate(assembler_inputs):
    print(i, col)

VectorAssembler inputs:
0 age_imputed
1 income_imputed
2 tenure_months_imputed
3 monthly_spend_imputed
4 website_visits_imputed
5 support_tickets_imputed
6 email_click_rate_imputed
7 loyalty_score_imputed
8 discount_used_imputed
9 last_campaign_days_imputed
10 gender_encoded
11 region_encoded
12 campaign_type_encoded


In [34]:
#  Final Logistic Regression coefficient analysis

for dataset_name, dataset_results in results.items():

    coefficients = (
        dataset_results["Logistic Regression"]["model"]
        .coefficients
        .toArray()
    )

    coefficient_df = pd.DataFrame({
        "Feature_Index": range(len(coefficients)),
        "Coefficient": coefficients,
        "Absolute_Importance": abs(coefficients)
    }).sort_values(
        "Absolute_Importance",
        ascending=False
    )

    print(f"\n===== {dataset_name} =====")
    display(coefficient_df)


===== Dataset 1 =====


,Feature_Index,Coefficient,Absolute_Importance
3,3,1.739645,1.739645
9,9,1.557306,1.557306
2,2,-0.710206,0.710206
6,6,-0.417495,0.417495
7,7,-0.374275,0.374275
4,4,-0.240244,0.240244
8,8,0.053020,0.053020
5,5,0.035633,0.035633
20,20,-0.017162,0.017162
14,14,-0.015372,0.015372



===== Dataset 2 =====


,Feature_Index,Coefficient,Absolute_Importance
3,3,4.250146,4.250146
9,9,1.600814,1.600814
2,2,-0.642994,0.642994
6,6,-0.403692,0.403692
7,7,-0.307817,0.307817
4,4,-0.295887,0.295887
8,8,0.115244,0.115244
5,5,0.040452,0.040452
0,0,0.037096,0.037096
1,1,-0.036172,0.036172


In [42]:
# Final model comparison table

final_results_df = (
    results_df
    .sort_values(["Dataset", "ROC_AUC"], ascending=[True, False])
    .reset_index(drop=True)
)

display(final_results_df)

,Dataset,Model,Accuracy,Precision,Recall,F1,ROC_AUC
0,Dataset 1,Logistic Regression,0.810740,0.810689,0.810740,0.810687,0.896021
1,Dataset 1,Gradient-Boosted Trees,0.803355,0.803305,0.803355,0.803314,0.888460
2,Dataset 1,Random Forest,0.794467,0.795334,0.794467,0.793902,0.875829
3,Dataset 2,Logistic Regression,0.939912,0.937465,0.939912,0.938208,0.968957
4,Dataset 2,Gradient-Boosted Trees,0.936132,0.933847,0.936132,0.934685,0.965763
5,Dataset 2,Random Forest,0.924990,0.923488,0.924990,0.914168,0.948949


In [38]:
final_results_df.to_csv(
    "HW3_final_model_results.csv",
    index=False
)

print("Saved successfully.")

Saved successfully.


## Final Results and Conclusion

Three Spark classification models were evaluated on both datasets:
Logistic Regression, Random Forest, and Gradient-Boosted Trees.

Logistic Regression achieved the best performance on both datasets.

- Dataset 1: ROC-AUC = 0.8960
- Dataset 2: ROC-AUC = 0.9690

Dataset 2 achieved substantially stronger predictive performance than
Dataset 1 across the evaluated metrics.

The results demonstrate that the Spark-based preprocessing and
classification pipeline can effectively predict customer churn.